In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

print("WEATHER DATA PROCESSING PIPELINE")
print("="*50)

print("\nSTEP 1: Generating weather data...")

dates = pd.date_range(start='2026-01-01', end='2026-06-24', freq='H')
cities = ['Milan', 'Rome', 'Venice', 'Turin', 'Naples']

weather_data = {
    'timestamp': np.repeat(dates, len(cities)),
    'city': cities * len(dates),
    'temperature': np.random.uniform(-10, 40, len(dates) * len(cities)),
    'humidity': np.random.uniform(20, 95, len(dates) * len(cities)),
    'wind_speed': np.random.uniform(0, 30, len(dates) * len(cities)),
    'precipitation': np.random.uniform(0, 100, len(dates) * len(cities)),
    'pressure': np.random.uniform(1000, 1030, len(dates) * len(cities))
}

df_weather = pd.DataFrame(weather_data)
print(f"Generated {len(df_weather)} records")
print(df_weather.head(10))

print("\nSTEP 2: Data validation and cleaning...")

null_count = df_weather.isnull().sum().sum()
print(f"Null values: {null_count}")

df_clean = df_weather.drop_duplicates(subset=['timestamp', 'city'])
print(f"After deduplication: {len(df_clean)} records")

df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'])
df_clean['temperature'] = df_clean['temperature'].clip(-50, 60)
df_clean['humidity'] = df_clean['humidity'].clip(0, 100)
df_clean['precipitation'] = df_clean['precipitation'].clip(0, 500)

print(f"Validation complete. Records: {len(df_clean)}")

print("\nSTEP 3: Feature engineering...")

df_clean = df_clean.sort_values(['city', 'timestamp'])

df_clean['temp_category'] = pd.cut(df_clean['temperature'], 
                                    bins=[-50, 0, 15, 25, 40, 60],
                                    labels=['Freezing', 'Cold', 'Mild', 'Warm', 'Hot'])

df_clean['is_rainy'] = (df_clean['precipitation'] > 5).astype(int)
df_clean['is_humid'] = (df_clean['humidity'] > 70).astype(int)

df_clean['hour'] = df_clean['timestamp'].dt.hour
df_clean['day'] = df_clean['timestamp'].dt.day
df_clean['month'] = df_clean['timestamp'].dt.month

df_clean['temp_ma_24h'] = df_clean.groupby('city')['temperature'].transform(
    lambda x: x.rolling(window=24, min_periods=1).mean()
)

print("Features added")
print(df_clean[['timestamp', 'city', 'temperature', 'temp_category', 'is_rainy']].head(10))

print("\nSTEP 4: Aggregations by city and time period...")

hourly_summary = df_clean.groupby(['timestamp', 'city']).agg({
    'temperature': ['mean', 'min', 'max'],
    'humidity': 'mean',
    'precipitation': 'sum',
    'wind_speed': 'mean',
    'is_rainy': 'sum'
}).reset_index()

hourly_summary.columns = ['timestamp', 'city', 'temp_mean', 'temp_min', 'temp_max', 
                          'humidity_mean', 'precipitation_total', 'wind_speed_mean', 'rainy_hours']

daily_summary = df_clean.groupby([df_clean['timestamp'].dt.date, 'city']).agg({
    'temperature': ['mean', 'min', 'max'],
    'humidity': 'mean',
    'precipitation': 'sum',
    'is_rainy': 'sum',
    'is_humid': 'sum'
}).reset_index()

print("Hourly Summary:")
print(hourly_summary.head())

print("\nDaily Summary:")
print(daily_summary.head())

print("\nSTEP 5: Monitoring and SLA checks...")

temp_anomalies = df_clean[df_clean['temperature'] > 45]
print(f"Temperature anomalies (>45C): {len(temp_anomalies)}")

high_precipitation = df_clean[df_clean['precipitation'] > 50]
print(f"High precipitation events (>50mm): {len(high_precipitation)}")

city_stats = df_clean.groupby('city').agg({
    'temperature': ['mean', 'std'],
    'humidity': 'mean',
    'precipitation': 'sum'
}).round(2)

print("\nCity Statistics:")
print(city_stats)

print("\nSTEP 6: Saving results...")

df_clean.to_csv('weather_data_cleaned.csv', index=False)
hourly_summary.to_csv('hourly_summary.csv', index=False)
daily_summary.to_csv('daily_summary.csv', index=False)

print("Files saved:")
print("- weather_data_cleaned.csv")
print("- hourly_summary.csv")
print("- daily_summary.csv")

print("\nSTEP 7: Quality Report...")

print(f"Total records processed: {len(df_clean)}")
print(f"Cities analyzed: {df_clean['city'].nunique()}")
print(f"Time range: {df_clean['timestamp'].min()} to {df_clean['timestamp'].max()}")
print(f"Missing values: {df_clean.isnull().sum().sum()}")
print(f"Data quality: {(len(df_clean)/len(df_weather))*100:.1f}%")

print("\nWeather pipeline completed successfully!")

WEATHER DATA PROCESSING PIPELINE

STEP 1: Generating weather data...
Generated 20885 records
            timestamp    city  temperature   humidity  wind_speed  \
0 2026-01-01 00:00:00   Milan    19.250069  27.187482    5.296461   
1 2026-01-01 00:00:00    Rome     4.272486  53.682409   14.816896   
2 2026-01-01 00:00:00  Venice     1.162707  22.304477    0.259218   
3 2026-01-01 00:00:00   Turin     8.309446  89.746758   11.979803   
4 2026-01-01 00:00:00  Naples    24.145210  59.152777   11.412373   
5 2026-01-01 01:00:00   Milan    17.979175  73.871559   22.349308   
6 2026-01-01 01:00:00    Rome    10.847626  42.963563   28.511792   
7 2026-01-01 01:00:00  Venice    15.456423  61.467747    1.885478   
8 2026-01-01 01:00:00   Turin    -0.625706  63.547222   26.421748   
9 2026-01-01 01:00:00  Naples    36.350713  85.160316    5.941148   

   precipitation     pressure  
0      72.446177  1018.926791  
1      63.364206  1029.158481  
2      21.473155  1011.245203  
3      35.123070  1

C:\Users\feder\AppData\Local\Temp\ipykernel_15896\2800922131.py:10: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  dates = pd.date_range(start='2026-01-01', end='2026-06-24', freq='H')


Files saved:
- weather_data_cleaned.csv
- hourly_summary.csv
- daily_summary.csv

STEP 7: Quality Report...
Total records processed: 20885
Cities analyzed: 5
Time range: 2026-01-01 00:00:00 to 2026-06-24 00:00:00
Missing values: 0
Data quality: 100.0%

Weather pipeline completed successfully!
